In [4]:
#  get the code (push local changes to GitHub first)
# delete any previous clone so every run starts from a fresh copy
%cd /content
!rm -rf /content/nepali-summarizer
!git clone https://github.com/tekraj/nepali-summarizer.git /content/nepali-summarizer
%cd /content/nepali-summarizer

/content
Cloning into '/content/nepali-summarizer'...
remote: Enumerating objects: 162335, done.
remote: Counting objects: 100% (9884/9884), done.
remote: Compressing objects: 100% (9869/9869), done.
remote: Total 162335 (delta 17), reused 9872 (delta 15), pack-reused 152451 (from 2)
Receiving objects: 100% (162335/162335), 148.80 MiB | 17.00 MiB/s, done.
Resolving deltas: 100% (2089/2089), done.
Updating files: 100% (160699/160699), done.
/content/nepali-summarizer


In [5]:
# install dependencies from pyproject.toml (keeps Colab's CUDA torch)
!pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 12.8 MB/s eta 0:00:00
  Building editable for nepali-summarizer (pyproject.toml) ... done


In [6]:
# data/cleaned is committed to git, so the clone above already includes it
!ls /content/nepali-summarizer/data/cleaned | wc -l

132563


In [7]:
# mount Drive so summaries survive a disconnect, then run the summarizer
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/nepali-summarizer")  # notebooks have no __file__
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.generate_summaries import run_summarizer

CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
SUMMARY_DIR = Path("/content/drive/MyDrive/nepali-summarizer/summary")
DEFAULT_MODEL = "GenzNepal/mt5-summarize-nepali"
DEFAULT_BATCH_SIZE = 16
DEFAULT_TOKENIZER_MAX_LEN = 4096
DEFAULT_GENERATE_MAX_LEN = 1024
DEFAULT_FORCE = False

run_summarizer(
    model = DEFAULT_MODEL,
    batch_size = DEFAULT_BATCH_SIZE,
    tokenizer_max_len = DEFAULT_TOKENIZER_MAX_LEN,
    generate_max_len = DEFAULT_GENERATE_MAX_LEN,
    cleaned_dir = CLEANED_DIR,
    summary_dir = SUMMARY_DIR,
    force = DEFAULT_FORCE,
)

Mounted at /content/drive
Total files provided: 132563
Files remaining to process: 132563

Loading model 'GenzNepal/mt5-summarize-nepali' onto device: cuda (Tesla T4)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

...

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]


Starting batch summarization...
Using batch_size=16, tokenizer_max_length=2048, generate_max_length=1024


/content/nepali-summarizer/src/data_preprocessing/summarizer.py:154: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast_ctx():


Processed 16/132563 files
Processed 400/132563 files


KeyboardInterrupt: 

In [ ]:
#  zip the summaries on Drive and download them to this computer
# Locally, unzip into data/summary:  unzip -n summary.zip -d data/summary
import shutil
from google.colab import files

ZIP_BASE = Path("/content/summary")  # make_archive adds the .zip extension
zip_path = shutil.make_archive(str(ZIP_BASE), "zip", root_dir=SUMMARY_DIR)
print(f"Zipped {len(list(SUMMARY_DIR.glob('*.txt')))} summaries -> {zip_path}")

shutil.copy(zip_path, SUMMARY_DIR.parent / "summary.zip")  # backup copy on Drive
files.download(zip_path)